In [ ]:

# Import the usual suspects.

from __future__ import print_function
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

#Import scikit-learn metrics module for accuracy calculation
from sklearn import metrics

from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from skmultilearn.problem_transform import BinaryRelevance, LabelPowerset
from skmultilearn.problem_transform import ClassifierChain

from sklearn.model_selection import GridSearchCV


import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('paper')



def print_ln():
    print('-' * 80, '\n')


In [ ]:
renamed_drug_columns_names = [                     'rifampicin_resistance',
                                                   'isoniazid_resistance',
                                                   'pyrazinamide_resistance',
                                                   'ethambutol_resistance',
                                                   'streptomycin_resistance',
                                                   'fluoroquinolones_resistance',
                                                   'moxifloxacin_resistance',
                                                   'ofloxacin_resistance',
                                                   'levofloxacin_resistance',
                                                   'ciprofloxacin_resistance',
                                                   'aminoglycosides_resistance',
                                                   'amikacin_resistance',
                                                   'kanamycin_resistance',
                                                   'capreomycin_resistance',
                                                   'ethionamide_resistance',
                                                   'para-aminosalicylic_acid_resistance',
                                                   'cycloserine_resistance',
                                                   'linezolid_resistance',
                                                   'bedaquiline_resistance',
                                                   'clofazimine_resistance',
                                                   'delamanid_resistance']


In [ ]:
multi_resistance_df_filledna = pd.read_csv("../data/processed/multi_resistance_filledna_df.csv")

multi_resistance_df_filledna.head()

In [ ]:
multi_resistance_df_filledna = multi_resistance_df_filledna.drop(columns= ['Unnamed: 0']).set_index('SampleID')
multi_resistance_df_filledna.head()



In [ ]:
# FIXME I've relied on manual conversion
# https://stackoverflow.com/questions/15891038/change-column-type-in-pandas#:~:text=The%20best%20way%20to%20convert%20one%20or%20more,Series%20or%20a%20single%20column%20of%20a%20DataFrame.

X = multi_resistance_df_filledna.drop(columns= renamed_drug_columns_names)
y = multi_resistance_df_filledna[renamed_drug_columns_names] \
                            .drop(columns= ['delamanid_resistance', 'cycloserine_resistance','linezolid_resistance' ], axis=1) 
#                             .astype(int) \
#                             .astype(str) \
#                             .astype('category') \

                            

In [ ]:
X

In [ ]:
X['NC000962_3.78']

In [ ]:
y


In [ ]:
y.describe().T


# Scikit-learn models

In [ ]:
# multi-label stratrification technique

from collections import Counter
from skmultilearn.model_selection.measures import get_combination_wise_output_matrix

from skmultilearn.dataset import load_dataset
X,y, _, _ = load_dataset('scene', 'undivided')

Counter(combination for row in get_combination_wise_output_matrix(y.A, order=2) for combination in row)



In [ ]:
X.shape

In [ ]:
_, original_y_train, _, _ = load_dataset('scene', 'train')
_, original_y_test, _, _ = load_dataset('scene', 'test')

In [ ]:
original_y_train

In [ ]:

pd.DataFrame({
    'train': Counter(str(combination) for row in get_combination_wise_output_matrix(original_y_train.A, order=2) for combination in row),
    'test' : Counter(str(combination) for row in get_combination_wise_output_matrix(original_y_test.A, order=2) for combination in row)
}).T.fillna(0.0)

In [ ]:
original_y_train

In [ ]:
# traditional splitting technique

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    train_size=0.7,
                                                    test_size=0.3,
                                                    random_state=100)


In [ ]:
from skmultilearn.model_selection import iterative_train_test_split

X_train, X_test, y_train, y_test = iterative_train_test_split(X, 
                                                              y,
                                                              test_size=0.3)


In [ ]:
X_train

In [ ]:
y_train

In [ ]:
Counter(combination for row in get_combination_wise_output_matrix(y_train, order=2) for combination in row)


In [ ]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# model_svm= BinaryRelevance(classifier= LinearSVC(), require_dense = [False, True])
model_svm= LabelPowerset(classifier= LinearSVC(), require_dense = [False, True])

# model_svm= ClassifierChain(classifier= SVC())

model_svm.fit(X_train, y_train)

y_pred= model_svm.predict(X_test)


# model_performance_metrics(model_svm, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
# print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
# print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
# print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

In [ ]:
print("Hamming loss:",metrics.hamming_loss(y_test, y_pred))

In [ ]:
model_svm

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from skmultilearn.cluster import LabelCooccurrenceGraphBuilder
from skmultilearn.ensemble import LabelSpacePartitioningClassifier

from sklearn.svm import SVC

parameters = {
    'classifier': [LabelPowerset(), BinaryRelevance()],
    'classifier__classifier': [SVC()]
}

clf = GridSearchCV(LabelSpacePartitioningClassifier(), parameters, scoring = 'f1_macro')
clf.fit(X_train, y_train)

print (clf.best_score_, clf.best_params_)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB

parameters = [
    {
        'classifier': [BernoulliNB()],
    },
    {
        'classifier': [MultinomialNB()],
        'classifier__alpha': [0.7, 1.0],
    },
    {
        'classifier': [SVC()],
        'classifier__kernel': ['rbf', 'linear'],
    },
]

# clf = GridSearchCV(BinaryRelevance(), parameters, scoring='accuracy')
clf = GridSearchCV(LabelPowerset(), parameters, scoring='accuracy')
clf.fit(X, y)

print(clf.best_params_, "\n", clf.best_score_,"\n", metrics.accuracy_score(y_test, y_pred))




In [ ]:
from sklearn.ensemble import RandomForestClassifier



# model_rf= BinaryRelevance(classifier= RandomForestClassifier(n_estimators= 100,
#                                   random_state = 100,
#                                   max_depth=5,
#                                   min_samples_leaf=50,
#                                   min_samples_split=50))


model_rf= LabelPowerset(classifier= RandomForestClassifier(n_estimators= 100,
                                  random_state = 100,
                                  max_depth=5,
                                  min_samples_leaf=50,
                                  min_samples_split=50),
                       require_dense = [False, False])



model_rf.fit(X_train, y_train)

y_pred= model_rf.predict(X_test)

# model_performance_metrics(model_rf, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# model_gb= BinaryRelevance(classifier=GradientBoostingClassifier(
#                                      n_estimators= 100,
#                                      random_state = 100,
#                                      max_depth=5
#                                     ),
#                          require_dense= [False, True])



model_gb= LabelPowerset(classifier=GradientBoostingClassifier(
                                     n_estimators= 100,
                                     random_state = 100,
                                     max_depth=5
                                    ),
                         require_dense= [False, False])


model_gb.fit(X_train, y_train)

y_pred= model_gb.predict(X_test)


# model_performance_metrics(model_gb, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB
# from sklearn.naive_bayes import BernoulliNB

# model_nb= BinaryRelevance(classifier= GaussianNB())
model_nb= ClassifierChain(classifier= GaussianNB())
# model_nb= BernoulliNB()

model_nb.fit(X_train, y_train)

y_pred= model_nb.predict(X_test)


# model_performance_metrics(model_nb, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.neural_network import MLPClassifier

model_mlp = BinaryRelevance(classifier= MLPClassifier(
                          solver='lbfgs',
                          alpha=1e-5,
                          hidden_layer_sizes=(5, 2),
                          random_state=1
))

# model_mlp = ClassifierChain(classifier= MLPClassifier(
#                           solver='lbfgs',
#                           alpha=1e-5,
#                           hidden_layer_sizes=(5, 2),
#                           random_state=1
# ))




model_mlp.fit(X_train, y_train)

y_pred= model_mlp.predict(X_test)

# model_performance_metrics(model_mlp, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from xgboost import XGBClassifier

model_xgb = BinaryRelevance(classifier= XGBClassifier(
                          learning_rate= 0.01,
                          random_state= 1
))

# model_xgb = ClassifierChain(classifier= XGBClassifier(
#                           learning_rate= 0.01,
#                           random_state= 1
# ))



model_xgb.fit(X_train, y_train)

y_pred= model_xgb.predict(X_test)

# model_performance_metrics(model_xgb, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier


estimators = [
    ('rf', RandomForestClassifier()),
    ('gb', GradientBoostingClassifier()),
    ('svc', LinearSVC()),
    ('mlp', MLPClassifier()),
    ('nb', GaussianNB()),
    ('xgb', XGBClassifier())
]

# model_se = BinaryRelevance(classifier= StackingClassifier(
#     estimators=estimators,
#     final_estimator=RandomForestClassifier()
# ))

model_se = ClassifierChain(classifier= StackingClassifier(
    estimators=estimators,
    final_estimator=RandomForestClassifier()
))

model_se.fit(X_train, y_train)

y_pred = model_se.predict(X_test)

# model_performance_metrics(model_se, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
Counter(combination for row in get_combination_wise_output_matrix(y_train, order=2) for combination in row)


In [ ]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# model_svm= BinaryRelevance(classifier= LinearSVC(), require_dense = [False, True])
model_svm= LabelPowerset(classifier= LinearSVC(), require_dense = [False, True])

# model_svm= ClassifierChain(classifier= SVC())

model_svm.fit(X_train, y_train)

y_pred= model_svm.predict(X_test)


# model_performance_metrics(model_svm, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
# print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
# print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
# print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

In [ ]:
print("Hamming loss:",metrics.hamming_loss(y_test, y_pred))

In [ ]:
model_svm

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from skmultilearn.cluster import LabelCooccurrenceGraphBuilder
from skmultilearn.ensemble import LabelSpacePartitioningClassifier

from sklearn.svm import SVC

parameters = {
    'classifier': [LabelPowerset(), BinaryRelevance()],
    'classifier__classifier': [SVC()]
}

clf = GridSearchCV(LabelSpacePartitioningClassifier(), parameters, scoring = 'f1_macro')
clf.fit(X_train, y_train)

print (clf.best_score_, clf.best_params_)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB

parameters = [
    {
        'classifier': [BernoulliNB()],
    },
    {
        'classifier': [MultinomialNB()],
        'classifier__alpha': [0.7, 1.0],
    },
    {
        'classifier': [SVC()],
        'classifier__kernel': ['rbf', 'linear'],
    },
]

# clf = GridSearchCV(BinaryRelevance(), parameters, scoring='accuracy')
clf = GridSearchCV(LabelPowerset(), parameters, scoring='accuracy')
clf.fit(X, y)

print(clf.best_params_, "\n", clf.best_score_,"\n", metrics.accuracy_score(y_test, y_pred))




In [ ]:
from sklearn.ensemble import RandomForestClassifier



# model_rf= BinaryRelevance(classifier= RandomForestClassifier(n_estimators= 100,
#                                   random_state = 100,
#                                   max_depth=5,
#                                   min_samples_leaf=50,
#                                   min_samples_split=50))


model_rf= LabelPowerset(classifier= RandomForestClassifier(n_estimators= 100,
                                  random_state = 100,
                                  max_depth=5,
                                  min_samples_leaf=50,
                                  min_samples_split=50),
                       require_dense = [False, False])



model_rf.fit(X_train, y_train)

y_pred= model_rf.predict(X_test)

# model_performance_metrics(model_rf, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# model_gb= BinaryRelevance(classifier=GradientBoostingClassifier(
#                                      n_estimators= 100,
#                                      random_state = 100,
#                                      max_depth=5
#                                     ),
#                          require_dense= [False, True])



model_gb= LabelPowerset(classifier=GradientBoostingClassifier(
                                     n_estimators= 100,
                                     random_state = 100,
                                     max_depth=5
                                    ),
                         require_dense= [False, False])


model_gb.fit(X_train, y_train)

y_pred= model_gb.predict(X_test)


# model_performance_metrics(model_gb, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB
# from sklearn.naive_bayes import BernoulliNB

# model_nb= BinaryRelevance(classifier= GaussianNB())
model_nb= ClassifierChain(classifier= GaussianNB())
# model_nb= BernoulliNB()

model_nb.fit(X_train, y_train)

y_pred= model_nb.predict(X_test)


# model_performance_metrics(model_nb, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.neural_network import MLPClassifier

model_mlp = BinaryRelevance(classifier= MLPClassifier(
                          solver='lbfgs',
                          alpha=1e-5,
                          hidden_layer_sizes=(5, 2),
                          random_state=1
))

# model_mlp = ClassifierChain(classifier= MLPClassifier(
#                           solver='lbfgs',
#                           alpha=1e-5,
#                           hidden_layer_sizes=(5, 2),
#                           random_state=1
# ))




model_mlp.fit(X_train, y_train)

y_pred= model_mlp.predict(X_test)

# model_performance_metrics(model_mlp, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from xgboost import XGBClassifier

model_xgb = BinaryRelevance(classifier= XGBClassifier(
                          learning_rate= 0.01,
                          random_state= 1
))

# model_xgb = ClassifierChain(classifier= XGBClassifier(
#                           learning_rate= 0.01,
#                           random_state= 1
# ))



model_xgb.fit(X_train, y_train)

y_pred= model_xgb.predict(X_test)

# model_performance_metrics(model_xgb, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier


estimators = [
    ('rf', RandomForestClassifier()),
    ('gb', GradientBoostingClassifier()),
    ('svc', LinearSVC()),
    ('mlp', MLPClassifier()),
    ('nb', GaussianNB()),
    ('xgb', XGBClassifier())
]

# model_se = BinaryRelevance(classifier= StackingClassifier(
#     estimators=estimators,
#     final_estimator=RandomForestClassifier()
# ))

model_se = ClassifierChain(classifier= StackingClassifier(
    estimators=estimators,
    final_estimator=RandomForestClassifier()
))

model_se.fit(X_train, y_train)

y_pred = model_se.predict(X_test)

# model_performance_metrics(model_se, X, X_test, X_train, y, y_test, y_pred, show_feature_importances= False)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

# Scikit-multilabel classifiers


In [ ]:
from skmultilearn.adapt import BRkNNaClassifier

model = BRkNNaClassifier(k=10)



model_xgb.fit(X_train, y_train)

y_pred= model_xgb.predict(X_test)

# model_performance_metrics(model_xgb, X, X_test, X_train, y, y_test, y_pred)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from skmultilearn.adapt import BRkNNaClassifier
from sklearn.model_selection import GridSearchCV

parameters = {'k': range(1,3)}
score = 'f1_macro'

clf = GridSearchCV(BRkNNaClassifier(), parameters, scoring=score)

clf.fit(X, y)

print(clf.best_params_, "\n", clf.best_score_,"\n", metrics.accuracy_score(y_test, y_pred))
